# EgoRecall — 03b: Deformable DETR Detection Results

**Evaluation-only notebook** — training ran on GCP VM via `train_detr.py`.

**Training details:**
- Stratified subset: 200 train clips (~18K frames), 50 val clips (~4.5K frames)
- 20 epochs on T4 GPU (~8 hours)
- Checkpoint saved every epoch — safe to resume if VM interrupted

**Results expected:**
| Model | mAP@0.5 | mAP@0.5:0.95 |
|-------|---------|-------------|
| Def-DETR zero-shot (COCO) | TBD | TBD |
| Def-DETR fine-tuned (Ego4D) | TBD | TBD |


## 0 · Imports & Config

In [ ]:
!pip install transformers torch torchvision google-cloud-storage \
    tqdm pandas pyarrow --quiet

In [ ]:
import io
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from tqdm.notebook import tqdm
from transformers import AutoImageProcessor, DeformableDetrForObjectDetection
from google.cloud import storage

BUCKET_NAME  = "egorecall-data"
DATA_DIR     = Path("/content/egorecall_detr")
RESULTS_DIR  = Path("/content/results/detr")
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DETR_MODEL   = "SenseTime/deformable-detr"
NUM_CLASSES  = 1
CONF_THRESH  = 0.25

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 1 · Auth & Download Weights

In [ ]:
from google.colab import auth
auth.authenticate_user()

gcs_client = storage.Client()
bucket     = gcs_client.bucket(BUCKET_NAME)
print(f"Connected to gs://{BUCKET_NAME}")

In [ ]:
# ── Download weights and metrics from GCS ─────────────────────────────────
os.makedirs("/content/models", exist_ok=True)

bucket.blob("models/detr_finetuned/best.pt").download_to_filename("/content/models/detr_best.pt")
bucket.blob("models/detr_finetuned/results.csv").download_to_filename("/content/models/detr_results.csv")

# Load subset clip list for downloading val frames
subset_data = json.loads(
    bucket.blob("processed/detr_subset_clips.json").download_as_bytes()
)
subset_val_clips = subset_data["val_clips"]
print(f"Downloaded weights and metrics.")
print(f"Val clips in subset: {len(subset_val_clips)}")

## 2 · Training Curves

In [ ]:
metrics_df = pd.read_csv("/content/models/detr_results.csv")
print(f"Epochs trained: {len(metrics_df)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(metrics_df["epoch"], metrics_df["train_loss"],
             color="#4C72B0", linewidth=2, label="Train")
axes[0].plot(metrics_df["epoch"], metrics_df["val_loss"],
             color="#DD8452", linewidth=2, label="Val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Curves", fontweight="bold")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Val loss improvement
axes[1].plot(metrics_df["epoch"],
             metrics_df["val_loss"] / metrics_df["val_loss"].iloc[0],
             color="#DD8452", linewidth=2)
axes[1].axhline(1.0, linestyle="--", color="gray", linewidth=1)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Val loss / initial val loss")
axes[1].set_title("Val Loss Improvement", fontweight="bold")
axes[1].grid(True, alpha=0.3)

plt.suptitle("Deformable DETR Fine-Tuning Curves", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "detr_training_curves.png", dpi=150)
plt.show()

best_epoch = metrics_df["val_loss"].idxmin()
print(f"Best epoch     : {metrics_df.loc[best_epoch, 'epoch']}")
print(f"Best val loss  : {metrics_df.loc[best_epoch, 'val_loss']:.4f}")

## 3 · Download Val Subset Frames

In [ ]:
# ── Download only the val clips used in training ───────────────────────────
val_img_dir = DATA_DIR / "val" / "images"
val_lbl_dir = DATA_DIR / "val" / "labels"
val_img_dir.mkdir(parents=True, exist_ok=True)
val_lbl_dir.mkdir(parents=True, exist_ok=True)

total_frames = 0
for clip_uid in tqdm(subset_val_clips, desc="Downloading val clips"):
    clip_img = val_img_dir / clip_uid
    clip_lbl = val_lbl_dir / clip_uid
    clip_img.mkdir(exist_ok=True)
    clip_lbl.mkdir(exist_ok=True)

    for blob in gcs_client.list_blobs(BUCKET_NAME,
                                       prefix=f"frames/detection/val/{clip_uid}/"):
        dst = clip_img / blob.name.split("/")[-1]
        if not dst.exists():
            blob.download_to_filename(str(dst))
        total_frames += 1

    for blob in gcs_client.list_blobs(BUCKET_NAME,
                                       prefix=f"labels/yolo/val/{clip_uid}/"):
        dst = clip_lbl / blob.name.split("/")[-1]
        if not dst.exists():
            blob.download_to_filename(str(dst))

print(f"Downloaded {total_frames:,} val frames across {len(subset_val_clips)} clips")

## 4 · Zero-Shot Evaluation

In [ ]:
def compute_iou(box1, box2):
    xi1 = max(box1[0], box2[0]); yi1 = max(box1[1], box2[1])
    xi2 = min(box1[2], box2[2]); yi2 = min(box1[3], box2[3])
    inter = max(0, xi2-xi1) * max(0, yi2-yi1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0


def compute_map(all_preds, all_gts, iou_thresholds=None):
    if iou_thresholds is None:
        iou_thresholds = np.arange(0.5, 1.0, 0.05)
    aps = []
    for iou_thresh in iou_thresholds:
        tp_list, fp_list, conf_list = [], [], []
        n_gt = sum(len(g) for g in all_gts)
        for preds, gts in zip(all_preds, all_gts):
            if not preds: continue
            matched = set()
            for pred in sorted(preds, key=lambda x: x[4], reverse=True):
                conf_list.append(pred[4])
                best_iou, best_idx = 0, -1
                for j, gt in enumerate(gts):
                    iou = compute_iou(pred[:4], gt)
                    if iou > best_iou: best_iou, best_idx = iou, j
                if best_iou >= iou_thresh and best_idx not in matched:
                    tp_list.append(1); fp_list.append(0); matched.add(best_idx)
                else:
                    tp_list.append(0); fp_list.append(1)
        if not conf_list: aps.append(0.0); continue
        order     = np.argsort(conf_list)[::-1]
        tp_cum    = np.cumsum(np.array(tp_list)[order])
        fp_cum    = np.cumsum(np.array(fp_list)[order])
        precision = tp_cum / (tp_cum + fp_cum + 1e-8)
        recall    = tp_cum / (n_gt + 1e-8)
        ap = sum(
            (precision[recall >= thr].max() if (recall >= thr).any() else 0)
            for thr in np.linspace(0, 1, 11)
        ) / 11
        aps.append(ap)
    return {"mAP@0.5": aps[0], "mAP@0.5:0.95": np.mean(aps)}


def detr_eval(model, processor, val_img_dir, val_lbl_dir,
               conf=CONF_THRESH, max_samples=2000):
    img_paths      = list(val_img_dir.rglob("*.jpg"))[:max_samples]
    all_pred_boxes = []
    all_gt_boxes   = []

    model.eval()
    with torch.no_grad():
        for img_path in tqdm(img_paths, desc="DETR eval"):
            lbl_path = val_lbl_dir / img_path.parent.name / f"{img_path.stem}.txt"
            if not lbl_path.exists(): continue

            img  = Image.open(img_path).convert("RGB")
            w, h = img.size
            gt_boxes = []
            for line in lbl_path.read_text().strip().split("\n"):
                if not line.strip(): continue
                _, cx, cy, bw, bh = map(float, line.split())
                gt_boxes.append([
                    (cx-bw/2)*w, (cy-bh/2)*h,
                    (cx+bw/2)*w, (cy+bh/2)*h
                ])

            inputs  = processor(images=img, return_tensors="pt").to(DEVICE)
            outputs = model(**inputs)
            results = processor.post_process_object_detection(
                outputs, threshold=conf,
                target_sizes=torch.tensor([[h, w]]).to(DEVICE)
            )[0]
            pred_boxes = [
                [*box.tolist(), score.item()]
                for score, box in zip(results["scores"], results["boxes"])
            ]
            all_pred_boxes.append(pred_boxes)
            all_gt_boxes.append(gt_boxes)

    return compute_map(all_pred_boxes, all_gt_boxes)


# Load pretrained model
detr_processor  = AutoImageProcessor.from_pretrained(DETR_MODEL)
detr_pretrained = DeformableDetrForObjectDetection.from_pretrained(DETR_MODEL).to(DEVICE)
detr_pretrained.eval()

print("Running zero-shot evaluation...")
detr_zeroshot = detr_eval(detr_pretrained, detr_processor, val_img_dir, val_lbl_dir)
print(f"\n── Def-DETR Zero-Shot ───────────────────────────────")
print(f"  mAP@0.5      : {detr_zeroshot['mAP@0.5']:.4f}")
print(f"  mAP@0.5:0.95 : {detr_zeroshot['mAP@0.5:0.95']:.4f}")
print("────────────────────────────────────────────────────")

## 5 · Fine-Tuned Evaluation

In [ ]:
# Load fine-tuned model
detr_finetuned_model = DeformableDetrForObjectDetection.from_pretrained(
    DETR_MODEL, num_labels=NUM_CLASSES, ignore_mismatched_sizes=True
).to(DEVICE)
detr_finetuned_model.load_state_dict(
    torch.load("/content/models/detr_best.pt", map_location=DEVICE)
)
detr_finetuned_model.eval()

print("Running fine-tuned evaluation...")
detr_finetuned = detr_eval(
    detr_finetuned_model, detr_processor, val_img_dir, val_lbl_dir
)
print(f"\n── Def-DETR Fine-Tuned ──────────────────────────────")
print(f"  mAP@0.5      : {detr_finetuned['mAP@0.5']:.4f}")
print(f"  mAP@0.5:0.95 : {detr_finetuned['mAP@0.5:0.95']:.4f}")
print("────────────────────────────────────────────────────")

## 6 · Results Summary

In [ ]:
results_df = pd.DataFrame([
    {"Model": "Def-DETR zero-shot (COCO)",  **detr_zeroshot},
    {"Model": "Def-DETR fine-tuned (Ego4D)", **detr_finetuned},
])
results_df["mAP@0.5"]      = results_df["mAP@0.5"].round(4)
results_df["mAP@0.5:0.95"] = results_df["mAP@0.5:0.95"].round(4)
print(results_df.to_string(index=False))

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ["#fcae91", "#cb181d"]
models = ["Zero-shot\n(COCO)", "Fine-tuned\n(Ego4D)"]

for ax, metric in zip(axes, ["mAP@0.5", "mAP@0.5:0.95"]):
    vals = results_df[metric].values
    bars = ax.bar(models, vals, color=colors, edgecolor="white", width=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.001,
                f"{val:.4f}", ha="center", va="bottom", fontweight="bold")
    ax.set_title(f"Def-DETR — {metric}", fontweight="bold")
    ax.set_ylabel(metric)
    ax.set_ylim(0, max(vals) * 1.3 + 0.01)

plt.suptitle("Deformable DETR: Zero-Shot vs Fine-Tuned (Ego4D VQ)",
             fontweight="bold", fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "detr_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# Save to GCS
results_df.to_parquet("/tmp/detr_results.parquet", index=False)
bucket.blob("processed/detr_results.parquet").upload_from_filename("/tmp/detr_results.parquet")
print(f"\nSaved to gs://{BUCKET_NAME}/processed/detr_results.parquet")
print("\nNote: DETR evaluated on subset (50 val clips).")
print("YOLOv8 evaluated on full val set (61,874 frames) — not directly comparable.")
print("Compare zero-shot→fine-tuned improvement within each model.")